# 09 · Đối chứng few-shot: QLoRA chỉ dạy "trả lời ngắn" hay dạy thêm được gì? (W7)

**Vấn đề:** B2 (QLoRA 1 epoch) tăng mạnh so với B1 (zero-shot): ViVQA EM 22,1 → 73,8; ViTextVQA EM 28,8 → 49,7. Chẩn đoán sau R1 lại cho thấy phần lớn lỗi của B1 là **định dạng**: B1 trả lời trung bình 5,5 từ, gold chỉ 3,7 từ. Trên ViTextVQA có 19% câu B1 "đọc ra đúng đáp án nhưng EM vẫn sai". Hội đồng sẽ hỏi: *"Chỉ cần đưa vài ví dụ vào prompt là đủ, cần gì fine-tune?"* Notebook này trả lời câu đó bằng thực nghiệm. Đây cũng là dòng **"zero-shot vs prompt-optimized vs QLoRA"** mà đề cương bắt buộc trong ablation.

**FS5 khác B1 đúng một biến: prompt có thêm 5 ví dụ (câu hỏi → đáp án), CHỈ CHỮ, lấy từ TRAIN.** Mọi thứ khác giữ y hệt:

| Điều kiện | B1 | FS5 (notebook này) | B2 |
|---|---|---|---|
| Model | Qwen2.5-VL-3B, 4-bit NF4, 512 token | **y hệt** | y hệt + LoRA |
| Prompt | P0: `Câu hỏi: …` | khối 5 ví dụ + `Câu hỏi: …\nTrả lời:` | P0 |
| Test, decoding, GPU | ViVQA 3.001 · ViTextVQA 2.000 · greedy · L4 | **y hệt** | y hệt |

**Chọn ví dụ (chốt trước, không chỉnh sau khi thấy kết quả):**
- k = 5, seed 42.
- Lấy từ train, chia đều theo `question_type`: ViVQA có đủ color, location, number, object; ViTextVQA chỉ có một loại nên là ngẫu nhiên.
- Đáp án dùng `answers[0]`, đúng như lúc train QLoRA.
- **Cùng 5 ví dụ cho mọi câu test.** Ví dụ ghi nguyên văn trong `run_signature.json`.

**Luật (chốt trước khi chạy), áp riêng cho từng dataset:**
1. **Few-shot có sửa được định dạng không:** có nếu EM(FS5) > EM(B1) **và** McNemar exact p < 0,05.
2. **QLoRA có lợi ích vượt ngoài định dạng không:** có nếu EM(B2) > EM(FS5) **và** McNemar exact p < 0,05. Đây là câu trả lời chính cho hội đồng.
3. **Tỉ lệ lợi ích giải thích được bằng prompt:** (EM_FS5 − EM_B1) / (EM_B2 − EM_B1). Kèm CI95 bootstrap ghép cặp (10.000 lần, seed 42). Chỉ báo cáo, không có ngưỡng.
4. **Chẩn đoán định dạng** (`src/eval/answer_format.py`): số từ, % "chứa đáp án gold", % "chứa gold nhưng EM sai". Chỉ để giải thích, không phải luật.

**Thời gian ước tính:** ViVQA ≈30 phút + ViTextVQA ≈30–35 phút → **≈1 giờ, ≈2 CU**. Chạy **sau** notebook 08 (một GPU một việc).

**Nếu Colab rớt:** chạy lại từ phần 1 rồi **Run all**. Các lô đã xong tự bỏ qua.

## 1. Thiết lập môi trường

Giống notebook 02/04/08 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (≥ commit thêm `--few_shot_k`), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B1/B2). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run, xem trước 5 ví dụ, kiểm tra trước khi tốn GPU

In ra đúng 5 ví dụ sẽ đưa vào prompt cho mỗi dataset. Chúng được chọn tất định từ train, nên chạy lại vẫn ra y hệt.

*Kỳ vọng: `bfloat16 | 401408 | nf4 | bfloat16`. ViVQA in 5 ví dụ gồm đủ màu, vị trí, số lượng, đồ vật. ViTextVQA in 5 ví dụ đọc chữ. B1/B2 có đủ 3001 và 2000 dự đoán. Hai dòng `SELFTEST OK`.*

In [ ]:
import json, yaml
from src.data.vivqa_dataset import load_vivqa
from src.eval.run_baseline import DATASETS, select_few_shot_examples

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT   = f'{DRIVE_ROOT}/experiments'
CHUNK, K, SEED = 500, 5, 42
OUT_DIR = f'{EXP_ROOT}/W07_FS_summary_L4_seed42'

D = {
    'vivqa': dict(ds='vivqa', n=3001,
                  fs='W07_FS5_fewshot_qwen25vl_vivqa_test_bf16_L4_seed42',
                  b1='W07_B1_zeroshot_qwen25vl_vivqa_test_bf16_L4_seed42',
                  b2='W07_B2_eval_qlora_r16_ep1_vivqa_test_bf16_L4_seed42'),
    'vitextvqa': dict(ds='vitextvqa_official', n=2000,
                  fs='W07_FS5_fewshot_qwen25vl_vitextvqa_test2000_bf16_L4_seed42',
                  b1='W07_B1_zeroshot_qwen25vl_vitextvqa_test2000_bf16_L4_seed42',
                  b2='W07_B2_eval_qlora_r16_ep1_vitextvqa_test2000_bf16_L4_seed42'),
}

cfg = yaml.safe_load(open('configs/qwen_lora.yaml', encoding='utf-8'))
mdl, q = cfg['model'], cfg['quantization']
print('torch_dtype | max_pixels | quant | compute dtype')
print(f"{mdl['torch_dtype']} | {mdl['max_pixels']} | {q.get('bnb_4bit_quant_type')} | {q.get('bnb_4bit_compute_dtype')}")
assert (mdl['torch_dtype'], mdl['max_pixels'], q.get('bnb_4bit_quant_type'), q.get('bnb_4bit_compute_dtype')) \
       == ('bfloat16', 401408, 'nf4', 'bfloat16'), 'Config KHÔNG phải bản chốt W07 -> dừng, báo Claude'

for key, d in D.items():
    for r in ('b1', 'b2'):
        n = len(json.load(open(f"{EXP_ROOT}/{d[r]}/predictions.json", encoding='utf-8')))
        assert n == d['n'], f'{key} {r}: {n} dự đoán, cần {d["n"]} -> dừng, báo Claude'
    train = load_vivqa(f"{DRIVE_ROOT}/{DATASETS[d['ds']]['train_rel']}", image_dir='')
    print(f"\n{key}: train {len(train)} mẫu | B1, B2 đủ {d['n']} dự đoán | 5 ví dụ few-shot:")
    for e in select_few_shot_examples(train, K, SEED):
        print(f"   Q: {e['question']}  ->  {e['answer']}")

!python -m src.eval.run_baseline --selftest 2>&1 | tail -n 1
!python -m src.eval.answer_format --selftest 2>&1 | tail -n 1

## 3. Chạy FS5 trên ViVQA rồi ViTextVQA (≈1 giờ)

Cùng script `run_baseline.py` như B1. Chỉ thêm `--few_shot_k 5 --few_shot_seed 42`. Run nào đã có `metrics.json` thì bỏ qua.

*Kỳ vọng: đầu mỗi run in lại 5 ví dụ, rồi 7 lô (ViVQA) hoặc 4 lô (ViTextVQA), cuối mỗi run `== KẾT QUẢ: EM=…`.*

In [ ]:
for key, d in D.items():
    run_name, ds = d['fs'], d['ds']          # biến đơn để IPython thay vào lệnh !python
    if os.path.exists(f'{EXP_ROOT}/{run_name}/metrics.json'):
        print(f'⏭ {key}: đã có metrics.json -> bỏ qua'); continue
    print(f'\n▶ {key}: {run_name}')
    !python -u -m src.eval.run_baseline --dataset {ds} --run_name {run_name} --chunk {CHUNK} --few_shot_k {K} --few_shot_seed {SEED}

## 4. Kết quả theo luật

Với mỗi dataset:
- **Bảng điểm** B1 / FS5 / B2.
- **Hai phép so ghép cặp:** B1 vs FS5 (luật 1) và FS5 vs B2 (luật 2).
- **Tỉ lệ lợi ích giải thích được bằng prompt** kèm CI95 (luật 3).
- **Bảng chẩn đoán định dạng.**

*Kỳ vọng: với mỗi dataset có 2 dòng `➜ LUẬT 1 …`, `➜ LUẬT 2 …` và 1 dòng `➜ Tỉ lệ …`.*

In [ ]:
import numpy as np
import pandas as pd
from src.eval.metrics import exact_match
from src.eval import answer_format as af

def load(run):
    d = f'{EXP_ROOT}/{run}'
    return (json.load(open(f'{d}/metrics.json', encoding='utf-8')),
            json.load(open(f'{d}/predictions.json', encoding='utf-8')))

def verdict(c, better, worse):
    if c['em_diff'] > 0 and c['p_value'] < 0.05:
        return better
    if c['em_diff'] < 0 and c['p_value'] < 0.05:
        return worse
    return 'CHƯA KẾT LUẬN ĐƯỢC (p ≥ 0,05)'

os.makedirs(OUT_DIR, exist_ok=True)
ROLE = {'B1': 'b1', 'FS5': 'fs', 'B2': 'b2'}
RES, PRED = {}, {}
for key, d in D.items():
    if not os.path.exists(f"{EXP_ROOT}/{d['fs']}/metrics.json"):
        print(f'{key}: chưa có FS5'); continue
    M = {lab: load(d[r]) for lab, r in (('B1', 'b1'), ('FS5', 'fs'), ('B2', 'b2'))}
    PRED[key] = {lab: p for lab, (_, p) in M.items()}
    rows = [{'run': lab, 'EM': 100 * m['metrics']['exact_match'], 'ANLS': 100 * m['metrics']['anls'],
             'F1 (phụ)': 100 * m['metrics'].get('token_f1', float('nan')), 's/câu': m['meta']['sec_per_sample']}
            for lab, (m, _) in M.items()]
    print(f'\n==================== {key} ====================')
    display(pd.DataFrame(rows).round(2))

    cmp = {}
    for a, b in (('B1', 'FS5'), ('FS5', 'B2')):
        out = f'{OUT_DIR}/paired_{a}_vs_{b}_{key}.json'
        ra, rb = f"{EXP_ROOT}/{d[ROLE[a]]}", f"{EXP_ROOT}/{d[ROLE[b]]}"
        !python src/eval/compare_runs.py --run_a {ra} --run_b {rb} --label_a {a} --label_b {b} --out {out}
        cmp[f'{a}_vs_{b}'] = json.load(open(out, encoding='utf-8'))
    v1 = verdict(cmp['B1_vs_FS5'], 'CÓ — few-shot tăng EM (p < 0,05)', 'KHÔNG — few-shot làm GIẢM EM (p < 0,05)')
    v2 = verdict(cmp['FS5_vs_B2'], 'CÓ — QLoRA vượt few-shot (p < 0,05)', 'KHÔNG — few-shot tốt hơn QLoRA (p < 0,05)')
    print(f"➜ LUẬT 1 ({key}) few-shot sửa được định dạng? {v1}")
    print(f"➜ LUẬT 2 ({key}) QLoRA có lợi ích ngoài định dạng? {v2}")

    ids = [p['question_id'] for p in PRED[key]['B1']]
    em = {}
    for lab, preds in PRED[key].items():
        byq = {x['question_id']: x for x in preds}
        em[lab] = np.array([exact_match(byq[q]['prediction'], byq[q]['answers']) for q in ids])
    share = (em['FS5'].mean() - em['B1'].mean()) / (em['B2'].mean() - em['B1'].mean())
    rng = np.random.default_rng(42)
    boots = []
    for _ in range(10000):
        i = rng.integers(0, len(ids), len(ids))
        den = em['B2'][i].mean() - em['B1'][i].mean()
        if den > 0:
            boots.append((em['FS5'][i].mean() - em['B1'][i].mean()) / den)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"➜ Tỉ lệ lợi ích của B2 mà prompt few-shot đạt được ({key}): {100 * share:.1f}% (CI95 [{100 * lo:.1f}; {100 * hi:.1f}])")

    fmt = {'gold': af.gold_summary(PRED[key]['B1']), **{lab: af.summarize(p) for lab, p in PRED[key].items()}}
    display(pd.DataFrame(fmt).T)
    pairs = {f'{a},{b}': af.paired_containment(PRED[key][a], PRED[key][b]) for a, b in (('B1', 'FS5'), ('FS5', 'B2'))}
    for k2, r in pairs.items():
        print(f"  chứa gold {k2}: Δ {r['delta_containment_pct']:+} điểm | McNemar p = {r['mcnemar_p']:.2g}")
    RES[key] = {'table': rows, 'rule1': v1, 'rule2': v2,
                'paired': {k2: {kk: vv for kk, vv in c.items() if kk != 'only_b_question_ids'} for k2, c in cmp.items()},
                'prompt_share_of_B2_gain': {'point': float(share), 'ci95': [float(lo), float(hi)], 'n_boot_valid': len(boots)},
                'answer_format': fmt, 'containment_pairs': pairs,
                'few_shot_examples': M['FS5'][0]['meta'].get('few_shot')}

## 5. Soi câu trả lời

Trên ViTextVQA, mỗi nhóm lấy 6 câu (seed 0):
- (a) few-shot **sửa đúng** câu B1 làm sai.
- (b) B2 đúng nhưng few-shot vẫn sai. Đây là phần QLoRA dạy thêm ngoài định dạng.

*Kỳ vọng: nhóm (a) chủ yếu là câu B1 trả lời dài; nhóm (b) cho thấy QLoRA học thêm gì (cách viết đáp án của dataset, đọc chữ, chọn vùng).*

In [ ]:
import random
key = 'vitextvqa' if 'vitextvqa' in PRED else next(iter(PRED))
by = {lab: {p['question_id']: p for p in PRED[key][lab]} for lab in PRED[key]}
ok = lambda lab, qid: exact_match(by[lab][qid]['prediction'], by[lab][qid]['answers'])
fixed = [q for q in by['B1'] if not ok('B1', q) and ok('FS5', q)]
beyond = [q for q in by['B1'] if ok('B2', q) and not ok('FS5', q)]
for title, group in [('(a) FEW-SHOT SỬA ĐÚNG', fixed), ('(b) CHỈ B2 ĐÚNG (ngoài định dạng)', beyond)]:
    print(f'\n######## {title}: {len(group)} câu ########')
    for q in random.Random(0).sample(group, min(6, len(group))):
        p = by['B1'][q]
        print(f"Q: {p['question']}\n   B1 : {p['prediction']!r}\n   FS5: {by['FS5'][q]['prediction']!r}\n"
              f"   B2 : {by['B2'][q]['prediction']!r}\n   GOLD: {p['answers'][:3]}")

## 6. Lưu tóm tắt

Ghi `experiments/W07_FS_summary_L4_seed42/fs_summary.json`. Cursor chép:
- File tóm tắt này và các file `paired_*.json` trong cùng thư mục.
- `metrics.json`, `predictions.json`, `run_signature.json` của hai run FS5.

**Không** chép `chunks/`.

*Kỳ vọng: `Ghi -> …/fs_summary.json`. Sau đó **Disconnect and delete runtime**.*

In [ ]:
summary = {
    'what': 'Đối chứng few-shot (FS5): B1 + 5 ví dụ chỉ chữ từ train trong prompt; tách lợi ích định dạng khỏi QLoRA (B2)',
    'pre_registered_rules': {
        'rule1': 'few-shot sửa định dạng nếu EM(FS5) > EM(B1) và McNemar exact p < 0,05',
        'rule2': 'QLoRA có lợi ích ngoài định dạng nếu EM(B2) > EM(FS5) và McNemar exact p < 0,05',
        'rule3': 'tỉ lệ = (EM_FS5 - EM_B1) / (EM_B2 - EM_B1), CI95 bootstrap ghép cặp 10.000 lần seed 42 (chỉ báo cáo)',
        'examples': 'k=5, seed 42, round-robin theo question_type, answers[0], cùng ví dụ cho mọi câu test'},
    'runs': {k: {lab: d[r] for lab, r in (('B1', 'b1'), ('FS5', 'fs'), ('B2', 'b2'))} for k, d in D.items()},
    'results': RES, 'gpu': GPU,
}
with open(f'{OUT_DIR}/fs_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2, default=str)
print('Ghi ->', f'{OUT_DIR}/fs_summary.json')
print('➡ Xong: Runtime -> Disconnect and delete runtime, rồi kiểm tra Manage sessions.')